# Dans Home Lakehouse Operations

Operations on the dans_home lakehouse using Delta Lake and pandas.

In [ ]:
# Import libraries
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from pathlib import Path
import os

In [ ]:
# Set up lakehouse path
lakehouse_path = r"C:\Users\danie\OneLake - Microsoft\MSTR_db\dans_home.Lakehouse\Tables"

# Verify path exists
if os.path.exists(lakehouse_path):
    print(f"✓ Lakehouse path found: {lakehouse_path}")
else:
    print(f"✗ Path not found: {lakehouse_path}")
    print("Please update the path to your dans_home lakehouse")

In [ ]:
# List all tables in the lakehouse
tables = [d for d in os.listdir(lakehouse_path) if os.path.isdir(os.path.join(lakehouse_path, d))]

print(f"Tables in dans_home lakehouse ({len(tables)}):")
for table in tables:
    table_path = os.path.join(lakehouse_path, table)
    # Check if it's a Delta table (has _delta_log)
    is_delta = os.path.exists(os.path.join(table_path, "_delta_log"))
    table_type = "Delta" if is_delta else "Parquet/Files"
    print(f"  - {table} ({table_type})")

## Create a New Delta Table

In [ ]:
# Create sample data
sample_data = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'department': ['Engineering', 'Sales', 'Marketing', 'Engineering', 'Sales'],
    'salary': [75000, 65000, 58000, 82000, 71000]
})

print("Sample data:")
sample_data

In [ ]:
# Write as Delta table
table_name = "employees"
table_path = os.path.join(lakehouse_path, table_name)

write_deltalake(
    table_path,
    sample_data,
    mode="overwrite",
    engine='pyarrow'
)

print(f"✓ Delta table created: {table_name}")
print(f"Location: {table_path}")

## Read Delta Tables

In [ ]:
# Read Delta table
dt = DeltaTable(table_path)
df = dt.to_pandas()

print(f"Table: {table_name}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nData:")
df

In [ ]:
# Get table metadata
print("Table Schema:")
print(dt.schema())

print("\nTable Version:")
print(dt.version())

print("\nTable History:")
history = dt.history()
for entry in history:
    print(f"  Version {entry['version']}: {entry['operation']} at {entry['timestamp']}")

## Append Data to Existing Table

In [ ]:
# Create new data to append
new_data = pd.DataFrame({
    'id': [6, 7],
    'name': ['Frank', 'Grace'],
    'department': ['Engineering', 'Marketing'],
    'salary': [79000, 62000]
})

# Append to Delta table
write_deltalake(
    table_path,
    new_data,
    mode="append",
    engine='pyarrow'
)

print("✓ Data appended")

# Read updated table
dt = DeltaTable(table_path)
df_updated = dt.to_pandas()
print(f"Total rows now: {len(df_updated)}")
df_updated

## Perform Data Analysis with Pandas

In [ ]:
# Group by department
dept_stats = df_updated.groupby('department').agg({
    'id': 'count',
    'salary': ['mean', 'min', 'max']
}).round(2)

dept_stats.columns = ['Employee_Count', 'Avg_Salary', 'Min_Salary', 'Max_Salary']
print("Department Statistics:")
dept_stats

In [ ]:
# Filter data
high_earners = df_updated[df_updated['salary'] > 70000]
print(f"Employees earning > $70,000: ({len(high_earners)} employees)")
high_earners

## Read Existing Table from Lakehouse

In [ ]:
# List tables again to choose one
tables = [d for d in os.listdir(lakehouse_path) 
          if os.path.isdir(os.path.join(lakehouse_path, d)) 
          and os.path.exists(os.path.join(lakehouse_path, d, "_delta_log"))]

print("Available Delta tables:")
for i, table in enumerate(tables, 1):
    print(f"{i}. {table}")

In [ ]:
# Read a specific table (change table_name as needed)
table_to_read = "employees"  # Change this to your table name
table_path_read = os.path.join(lakehouse_path, table_to_read)

if os.path.exists(table_path_read):
    dt_read = DeltaTable(table_path_read)
    df_read = dt_read.to_pandas()
    
    print(f"Table: {table_to_read}")
    print(f"Rows: {len(df_read)}")
    print(f"Columns: {list(df_read.columns)}")
    print("\nFirst 10 rows:")
    display(df_read.head(10))
else:
    print(f"Table '{table_to_read}' not found")

## Time Travel - Read Historical Versions

In [ ]:
# Read specific version of the table
dt = DeltaTable(table_path)

# Read version 0 (original data)
df_v0 = dt.to_pandas(version=0)

print(f"Version 0 had {len(df_v0)} rows")
df_v0

## Export Table to CSV

In [ ]:
# Export to CSV
export_path = os.path.join(os.path.dirname(lakehouse_path), "Files", f"{table_name}_export.csv")

df_updated.to_csv(export_path, index=False, encoding='utf-8')
print(f"✓ Exported to: {export_path}")

## Load CSV to Delta Table

In [ ]:
# Load CSV from Files folder
files_path = os.path.join(os.path.dirname(lakehouse_path), "Files")

# List CSV files
csv_files = [f for f in os.listdir(files_path) if f.endswith('.csv')]
print(f"CSV files in Files folder: {csv_files}")

# Example: Load a CSV and convert to Delta table
if csv_files:
    csv_file = csv_files[0]  # Take first CSV
    csv_path = os.path.join(files_path, csv_file)
    
    df_from_csv = pd.read_csv(csv_path)
    
    # Write as Delta table
    new_table_name = csv_file.replace('.csv', '')
    new_table_path = os.path.join(lakehouse_path, new_table_name)
    
    write_deltalake(
        new_table_path,
        df_from_csv,
        mode="overwrite",
        engine='pyarrow'
    )
    
    print(f"✓ Created Delta table '{new_table_name}' from CSV")
    print(f"Rows: {len(df_from_csv)}")
    df_from_csv.head()